# 02 — SEER Cancer Registry: County-Level Cancer Incidence

**Data source:** NCI SEER Public Data Files (no API key required)  
**What it contains:** County-level breast cancer incidence rates and case counts  
**Geographic grain:** County (FIPS) — joins directly onto our CMS provider table  
**DuckDB table written:** `seer_cancer`  
**Why it matters:** Secondary lymphedema is caused by cancer treatment in ~60% of cases.
Breast cancer survivors have a 20-30% lifetime risk of developing lymphedema.
High breast cancer incidence counties = high future compression garment demand,
regardless of whether patients are on Medicare or privately insured.



# SEER Cancer Registry
**Data Source:** SEER Cancer Registry

**Output:** County-level cancer incidence and survivorship by type

**DuckDB Table:** `seer_cancer`



## Setup

In [1]:
import sys
import pandas as pd
import requests
import zipfile
import io
from pathlib import Path

sys.path.append(str(Path('..') / 'src'))

from config import DATA_RAW, DATA_PROCESSED, DB_PATH, TARGET_CODES
from db import get_connection
from utils import ensure_dirs, log

ensure_dirs()
log("Notebook 02 — SEER Cancer — started")

[2026-04-19 19:02:56] INFO - Notebook 02 — SEER Cancer — started


## Step 1 — Data Collection
Download county-level cancer incidence and survivorship data by type from SEER.

In [3]:
# NCI State Cancer Profiles provides county-level cancer incidence data
# as downloadable CSVs — no authentication required.
# We query their data export API which returns CSV data for any cancer type
# filtered by state, county, and cancer site.
#
# Base URL for State Cancer Profiles data export
# incidence = new cases per 100,000 population (age-adjusted rate)

BASE_URL = "https://statecancerprofiles.cancer.gov/incidencerates/index.php"

# Test with a single request — breast cancer, all races, all ages, county level
# Parameters:
#   stateFIPS = 00 means all states
#   cancer    = 055 is the NCI code for female breast cancer
#   race      = 00 = all races
#   sex       = 2 = female
#   age       = 001 = all ages
#   stage     = 999 = all stages
#   year      = 0 = latest available
#   type      = incd = incidence
#   sortVariableName = rate
#   output    = 1 = CSV format

params = {
    "stateFIPS": "00",
    "areatype":  "county",
    "cancer":    "055",
    "race":      "00",
    "sex":       "2",
    "age":       "001",
    "stage":     "999",
    "year":      "0",
    "type":      "incd",
    "sortVariableName": "rate",
    "output":    "1",
}

log("Testing NCI State Cancer Profiles export...")
resp = requests.get(BASE_URL, params=params, timeout=30)
print("Status code:", resp.status_code)
print("First 500 chars of response:")
print(resp.text[:500])

[2026-04-19 19:04:38] INFO - Testing NCI State Cancer Profiles export...
Status code: 200
First 500 chars of response:
Incidence Rate Report for United States by County

"Breast (All Stages^), 2018-2022"

"All Races (includes Hispanic), Female, All Ages"

Sorted by Rate

County,FIPS,2023 Rural-Urban Continuum Codes([rural urban note]),"Age-Adjusted Incidence Rate([rate note]) - cases per 100,000","Lower 95% Confidence Interval","Upper 95% Confidence Interval","CI*Rank([rank note])","Lower CI (CI*Rank)","Upper CI (CI*Rank)",Average Annual Count,Recent Trend,"Recent 5-Year Trend ([trend note]) in Incidence Rates",


In [4]:
# Parse the CSV response — NCI includes header rows we need to skip
# The actual data starts after the metadata lines at the top

lines = resp.text.split('\n')

# Find the line index where the actual column headers start
# We look for the line that starts with "County,FIPS"
header_idx = next(i for i, line in enumerate(lines) if line.startswith('County,FIPS'))

print(f"Header row found at line index: {header_idx}")
print(f"Header: {lines[header_idx]}")
print(f"First data row: {lines[header_idx + 1]}")

Header row found at line index: 8
Header: County,FIPS,2023 Rural-Urban Continuum Codes([rural urban note]),"Age-Adjusted Incidence Rate([rate note]) - cases per 100,000","Lower 95% Confidence Interval","Upper 95% Confidence Interval","CI*Rank([rank note])","Lower CI (CI*Rank)","Upper CI (CI*Rank)",Average Annual Count,Recent Trend,"Recent 5-Year Trend ([trend note]) in Incidence Rates","Lower 95% Confidence Interval","Upper 95% Confidence Interval"
First data row: "US (SEER+NPCR)(1)",00000,N/A,131.3 ,131.1, 131.6,N/A , N/A , N/A,270245,rising,0.6 ,0.5, 0.8


In [5]:
# Load CSV from line 8 onwards, skip the metadata header rows
import io

seer_raw = pd.read_csv(
    io.StringIO(resp.text),
    skiprows=header_idx,
    dtype=str,
)

print(f"Raw rows: {len(seer_raw):,}")
print(f"Columns: {seer_raw.columns.tolist()}")
seer_raw.head(3)

Raw rows: 3,161
Columns: ['County', 'FIPS', '2023 Rural-Urban Continuum Codes([rural urban note])', 'Age-Adjusted Incidence Rate([rate note]) - cases per 100,000', 'Lower 95% Confidence Interval', 'Upper 95% Confidence Interval', 'CI*Rank([rank note])', 'Lower CI (CI*Rank)', 'Upper CI (CI*Rank)', 'Average Annual Count', 'Recent Trend', 'Recent 5-Year Trend ([trend note]) in Incidence Rates', 'Lower 95% Confidence Interval.1', 'Upper 95% Confidence Interval.1']


,County,FIPS,2023 Rural-Urban Continuum Codes([rural urban note]),"Age-Adjusted Incidence Rate([rate note]) - cases per 100,000",Lower 95% Confidence Interval,Upper 95% Confidence Interval,CI*Rank([rank note]),Lower CI (CI*Rank),Upper CI (CI*Rank),Average Annual Count,Recent Trend,Recent 5-Year Trend ([trend note]) in Incidence Rates,Lower 95% Confidence Interval.1,Upper 95% Confidence Interval.1
0,US (SEER+NPCR)(1),00000,NaN,131.3,131.1,131.6,N/A,N/A,N/A,270245,rising,0.6,0.5,0.8
1,"Aurora County, South Dakota(2)",46003,Rural,288.8,176.1,447.8,N/A,N/A,N/A,5,stable,2.9,-2.4,8.3
2,"Armstrong County, Texas(7)",48011,Urban,271.9,144.5,467.4,N/A,N/A,N/A,3,*,*,*,*


In [6]:
# Keep and rename only the columns we need
seer = seer_raw[['County', 'FIPS', 'Age-Adjusted Incidence Rate([rate note]) - cases per 100,000', 'Average Annual Count']].copy()

seer = seer.rename(columns={
    'County':       'county_name',
    'FIPS':         'fips_county',
    'Age-Adjusted Incidence Rate([rate note]) - cases per 100,000': 'breast_cancer_rate',
    'Average Annual Count': 'breast_cancer_avg_annual_cases',
})

# Remove the national summary row (FIPS = 00000) and any state-level summary rows
# We only want county-level rows — those have 5-digit FIPS ending in non-zero last 3 digits
seer = seer[seer['fips_county'].str.strip() != '00000']
seer = seer[seer['fips_county'].str.len() == 5]

# Clean FIPS — strip whitespace
seer['fips_county'] = seer['fips_county'].str.strip()

# Cast numerics — replace suppressed values ("* " = fewer than 16 cases, not shown)
# We replace suppressed with NaN — they are small rural counties, low priority anyway
for col in ['breast_cancer_rate', 'breast_cancer_avg_annual_cases']:
    seer[col] = pd.to_numeric(seer[col].str.strip(), errors='coerce')

# Drop rows where both rate and count are missing
before = len(seer)
seer = seer.dropna(subset=['breast_cancer_rate', 'breast_cancer_avg_annual_cases'], how='all')
log(f"Dropped {before - len(seer):,} fully suppressed county rows")

log(f"Clean rows: {len(seer):,} counties")
seer.head(5)

[2026-04-19 19:05:44] INFO - Dropped 338 fully suppressed county rows
[2026-04-19 19:05:44] INFO - Clean rows: 2,805 counties


,county_name,fips_county,breast_cancer_rate,breast_cancer_avg_annual_cases
1,"Aurora County, South Dakota(2)",46003,288.8,5.0
2,"Armstrong County, Texas(7)",48011,271.9,3.0
3,"Haakon County, South Dakota(2)",46055,236.6,3.0
4,"Stanley County, South Dakota(2)",46117,235.8,4.0
5,"McCook County, South Dakota(2)",46087,230.2,8.0


In [7]:
# Strip footnote markers from county names e.g. "Aurora County, South Dakota(2)" → "Aurora County, South Dakota"
seer['county_name'] = seer['county_name'].str.replace(r'\(\d+\)', '', regex=True).str.strip()

# Save raw to disk
raw_out = DATA_RAW['seer'] / 'seer_breast_cancer_county_raw.csv'
seer_raw.to_csv(raw_out, index=False)
log(f"Raw saved → {raw_out}")

# Save processed to disk
processed_out = DATA_PROCESSED['seer'] / 'seer_breast_cancer_county_clean.csv'
seer.to_csv(processed_out, index=False)
log(f"Processed saved → {processed_out}")

# Write to DuckDB
con = get_connection()
con.execute("DROP TABLE IF EXISTS seer_cancer")
con.execute("CREATE TABLE seer_cancer AS SELECT * FROM seer")
count = con.execute("SELECT COUNT(*) FROM seer_cancer").fetchone()[0]
log(f"seer_cancer written to DuckDB: {count:,} rows")
con.close()

seer.head(5)

[2026-04-19 19:06:48] INFO - Raw saved → C:\Users\sanat\Desktop\AntiGravity\findingclaims\data\raw\seer\seer_breast_cancer_county_raw.csv
[2026-04-19 19:06:48] INFO - Processed saved → C:\Users\sanat\Desktop\AntiGravity\findingclaims\data\processed\seer\seer_breast_cancer_county_clean.csv
[2026-04-19 19:06:48] INFO - seer_cancer written to DuckDB: 2,805 rows


,county_name,fips_county,breast_cancer_rate,breast_cancer_avg_annual_cases
1,"Aurora County, South Dakota",46003,288.8,5.0
2,"Armstrong County, Texas",48011,271.9,3.0
3,"Haakon County, South Dakota",46055,236.6,3.0
4,"Stanley County, South Dakota",46117,235.8,4.0
5,"McCook County, South Dakota",46087,230.2,8.0


In [8]:
# ── QA & summary stats ──────────────────────────────────────────────────────
con = get_connection()

print("=== Top 10 counties by average annual breast cancer cases ===")
print(con.execute("""
    SELECT 
        county_name,
        fips_county,
        breast_cancer_rate          AS rate_per_100k,
        breast_cancer_avg_annual_cases AS avg_annual_cases
    FROM seer_cancer
    ORDER BY breast_cancer_avg_annual_cases DESC
    LIMIT 10
""").df().to_string(index=False))

print("\n=== Top 10 counties by incidence rate per 100k ===")
print(con.execute("""
    SELECT 
        county_name,
        fips_county,
        breast_cancer_rate          AS rate_per_100k,
        breast_cancer_avg_annual_cases AS avg_annual_cases
    FROM seer_cancer
    ORDER BY breast_cancer_rate DESC
    LIMIT 10
""").df().to_string(index=False))

print("\n=== Counties we can join to CMS providers ===")
print(con.execute("""
    SELECT COUNT(DISTINCT s.fips_county) AS seer_counties_in_cms
    FROM seer_cancer s
    INNER JOIN cms_providers c ON s.fips_county = c.fips_county
""").df().to_string(index=False))

con.close()

=== Top 10 counties by average annual breast cancer cases ===
                   county_name fips_county  rate_per_100k  avg_annual_cases
Los Angeles County, California       06037          122.9            7174.0
         Cook County, Illinois       17031          131.0            4138.0
      Maricopa County, Arizona       04013          122.4            3240.0
          Harris County, Texas       48201          120.4            2869.0
     Orange County, California       06059          132.3            2549.0
  San Diego County, California       06073          130.3            2452.0
                   Puerto Rico       72001           98.9            2410.0
    Miami-Dade County, Florida       12086          116.6            2056.0
        Kings County, New York       36047          125.9            1980.0
       Queens County, New York       36081          126.5            1914.0

=== Top 10 counties by incidence rate per 100k ===
                      county_name fips_county  rat

## Notebook 02 complete ✓

### What we built
- Pulled county-level breast cancer incidence data for all US counties from NCI State Cancer Profiles
- 2,805 counties with clean rate and case count data (2018-2022, age-adjusted)
- 338 suppressed counties dropped (fewer than 16 cases — tiny rural counties)
- Written to DuckDB table: `seer_cancer`
- Saved to `data/processed/seer/`

### Key findings
- Highest volume counties: LA, Cook (Chicago), Maricopa (Phoenix), Harris (Houston)
- 2,024 counties overlap with our CMS provider data — our combined targeting universe
- Rate alone is misleading — always use average annual cases for sales prioritization

### Important distinction
- High rate + high cases = priority sales territory
- High rate + low cases = small rural population, deprioritize
- Low rate + high cases = large urban population, still worth targeting

### Next notebook
`03_cdc_places.ipynb` — county-level chronic disease prevalence  
Adds obesity, diabetes, and cardiovascular disease burden by county —  
comorbidities that drive chronic venous disease and lymphedema demand.